In [ ]:
!pip install category_encoders scikit-learn pandas torch numpy tomli_w tomli zero

# %%
import math
import typing as ty
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.init as nn_init
from torch import Tensor

import lib


def attenuated_kaiming_uniform_(tensor, a=math.sqrt(5), scale=1., mode='fan_in', nonlinearity='leaky_relu'):
    fan = nn_init._calculate_correct_fan(tensor, mode)
    gain = nn_init.calculate_gain(nonlinearity, a)
    std = gain * scale / math.sqrt(fan)
    bound = math.sqrt(3.0) * std  # Calculate uniform bounds from standard deviation
    with torch.no_grad():
        return tensor.uniform_(-bound, bound)

# def attenuated_init(tensor: Tensor, activation='none', scale=1.):
#     if activation in ['relu', 'tanh', 'sigmoid']:
#         nn_init.xavier_uniform_(tensor, gain=nn_init.calculate_gain(activation) * scale)
#     elif activation == 'attention':
#         nn_init.xavier_uniform_(tensor, gain=1 / math.sqrt(2) * scale)
#     else:
#         nn_init.xavier_uniform_(tensor, gain=1.0 * scale)


class Tokenizer(nn.Module):
    category_offsets: ty.Optional[Tensor]

    def __init__(
        self,
        d_numerical: int,
        categories: ty.Optional[ty.List[int]],
        d_token: int,
        bias: bool,
    ) -> None:
        super().__init__()
        print(f"d_numerical: {d_numerical}")
        print(f"categories: {categories}")
        print(f"d_token: {d_token}")
        print(f"bias: {bias}")

        if categories is None:
            d_bias = d_numerical
            self.category_offsets = None
            self.category_embeddings = None
        else:
            d_bias = d_numerical + len(categories)
            category_offsets = torch.tensor([0] + categories[:-1]).cumsum(0)
            self.register_buffer('category_offsets', category_offsets)
            self.category_embeddings = nn.Embedding(sum(categories), d_token)
            nn_init.kaiming_uniform_(self.category_embeddings.weight, a=math.sqrt(5))
            print(f'{self.category_embeddings.weight.shape}')

        print(f'category_embeddings.weight.shape: {self.category_embeddings.weight.shape}')
        print(f"d_bias: {d_bias}")

        self.weight = nn.Parameter(Tensor(d_numerical, d_token))
        self.weight2 = nn.Parameter(Tensor(d_numerical, d_token))
        self.bias = nn.Parameter(Tensor(d_bias, d_token)) if bias else None
        self.bias2 = nn.Parameter(Tensor(d_bias, d_token)) if bias else None

        print(f"weight.shape: {self.weight.shape}")
        print(f"weight2.shape: {self.weight2.shape}")
        print(f"bias.shape: {self.bias.shape if self.bias is not None else None}")
        print(f"bias2.shape: {self.bias2.shape if self.bias2 is not None else None}")

        # v4
        attenuated_kaiming_uniform_(self.weight)
        attenuated_kaiming_uniform_(self.weight2)
        nn_init.kaiming_uniform_(self.bias, a=math.sqrt(5))
        nn_init.kaiming_uniform_(self.bias2, a=math.sqrt(5))


    @property
    def n_tokens(self) -> int:
        return len(self.weight) + (
            0 if self.category_offsets is None else len(self.category_offsets)
        )

    def forward(self, x_num: Tensor) -> Tensor:
        x_some = x_num
        assert x_some is not None
        print(f"x_num shape: {x_num.shape}")
        print(f"self.weight shape: {self.weight.shape}")
        print(f"self.bias shape: {self.bias.shape}")
        x1 = self.weight[None] * x_num[:, :, None] + self.bias[None]
        x2 = self.weight2[None] * x_num[:, :, None] + self.bias2[None]
        return x1 * torch.tanh(x2)


class MultiheadAttention(nn.Module):
    def __init__(
        self, d: int, n_heads: int, dropout: float, init_scale: float = 0.01
    ) -> None:
        if n_heads > 1:
            assert d % n_heads == 0
        # assert initialization in ['xavier', 'kaiming']

        super().__init__()
        self.W_q = nn.Linear(d, d)
        self.W_k = nn.Linear(d, d)
        self.W_v = nn.Linear(d, d)
        self.W_out = nn.Linear(d, d) if n_heads > 1 else None
        self.n_heads = n_heads
        self.dropout = nn.Dropout(dropout) if dropout else None

        for i, m in enumerate([self.W_q, self.W_k, self.W_v]):
            # all small
            attenuated_kaiming_uniform_(m.weight, scale=init_scale)
            nn_init.zeros_(m.bias)
        if self.W_out is not None:
            attenuated_kaiming_uniform_(self.W_out.weight)
            nn_init.zeros_(self.W_out.bias)

    def _reshape(self, x: Tensor) -> Tensor:
        batch_size, n_tokens, d = x.shape
        d_head = d // self.n_heads
        return (
            x.reshape(batch_size, n_tokens, self.n_heads, d_head)
            .transpose(1, 2)
            .reshape(batch_size * self.n_heads, n_tokens, d_head)
        )

    def get_attention_mask(self, input_shape, device):
        bs, _, seq_len = input_shape
        seq_ids = torch.arange(seq_len, device=device)
        attention_mask = seq_ids[None, None, :].repeat(bs, seq_len, 1) <= seq_ids[None, :, None]
        # attention_mask = seq_ids[None, :].repeat(seq_len, 1) <= seq_ids[:, None]
        attention_mask = (1.0 - attention_mask.float()) * -1e4
        return attention_mask

    def forward(
        self,
        x_q: Tensor,
        x_kv: Tensor,
        key_compression: ty.Optional[nn.Linear],
        value_compression: ty.Optional[nn.Linear],
    ) -> Tensor:
        q, k, v = self.W_q(x_q), self.W_k(x_kv), self.W_v(x_kv)
        for tensor in [q, k, v]:
            assert tensor.shape[-1] % self.n_heads == 0
        if key_compression is not None:
            assert value_compression is not None
            k = key_compression(k.transpose(1, 2)).transpose(1, 2)
            v = value_compression(v.transpose(1, 2)).transpose(1, 2)
        else:
            assert value_compression is None

        batch_size = len(q)
        d_head_key = k.shape[-1] // self.n_heads
        d_head_value = v.shape[-1] // self.n_heads
        n_q_tokens = q.shape[1]

        q = self._reshape(q)
        k = self._reshape(k)
        attention_scores = q @ k.transpose(1, 2) / math.sqrt(d_head_key) # b f f
        masks = self.get_attention_mask(attention_scores.shape, attention_scores.device)
        attention = F.softmax(attention_scores + masks, dim=-1)
        if self.dropout is not None:
            attention = self.dropout(attention)
        x = attention @ self._reshape(v)
        x = (
            x.reshape(batch_size, self.n_heads, n_q_tokens, d_head_value)
            .transpose(1, 2)
            .reshape(batch_size, n_q_tokens, self.n_heads * d_head_value)
        )
        if self.W_out is not None:
            x = self.W_out(x)
        return x


class ExcelFormer(nn.Module):
    """ExcelFormer with All initialized by small value

    initial function: v4
    """
    def __init__(
        self,
        *,
        # tokenizer
        d_numerical: int,
        categories: ty.Optional[ty.List[int]],
        token_bias: bool,
        # transformer
        n_layers: int,
        d_token: int,
        n_heads: int,
        attention_dropout: float,
        ffn_dropout: float,
        residual_dropout: float,
        prenormalization: bool,
        # linformer
        kv_compression: ty.Optional[float],
        kv_compression_sharing: ty.Optional[str],
        #
        d_out: int,
        init_scale: float = 0.1,
    ) -> None:
        assert (kv_compression is None) ^ (kv_compression_sharing is not None)

        super().__init__()
        # assert initialization in ['xavier', 'kaiming']
        n_tokens = d_numerical + len(categories) if categories is not None else d_numerical
        self.tokenizer = Tokenizer(d_numerical, categories, d_token, token_bias)
        self.n_categories = 0 if categories is None else len(categories)

        def make_kv_compression():
            assert kv_compression
            compression = nn.Linear(
                n_tokens, int(n_tokens * kv_compression), bias=False
            )
            return compression

        self.shared_kv_compression = (
            make_kv_compression()
            if kv_compression and kv_compression_sharing == 'layerwise'
            else None
        )

        def make_normalization():
            return nn.LayerNorm(d_token)

        self.layers = nn.ModuleList([])
        for layer_idx in range(n_layers):
            layer = nn.ModuleDict(
                {
                    # Attenuated Initialization
                    'attention': MultiheadAttention(
                        d_token, n_heads, attention_dropout, init_scale=init_scale
                    ),
                    'linear0': nn.Linear(d_token, d_token * 2),
                    'norm1': make_normalization(),
                }
            )
            # Attenuated Initialization
            attenuated_kaiming_uniform_(layer['linear0'].weight, scale=init_scale)
            nn_init.zeros_(layer['linear0'].bias)

            if not prenormalization or layer_idx:
                layer['norm0'] = make_normalization()
            if kv_compression and self.shared_kv_compression is None:
                layer['key_compression'] = make_kv_compression()
                if kv_compression_sharing == 'headwise':
                    layer['value_compression'] = make_kv_compression()
                else:
                    assert kv_compression_sharing == 'key-value'
            self.layers.append(layer)

        self.activation = lib.get_activation_fn('tanglu')
        self.last_activation = nn.PReLU()
        self.prenormalization = prenormalization
        self.last_normalization = make_normalization() if prenormalization else None
        self.ffn_dropout = ffn_dropout
        self.residual_dropout = residual_dropout

        # Attenuated Initialization
        self.head = nn.Linear(d_token, d_out)
        attenuated_kaiming_uniform_(self.head.weight)
        # nn_init.zeros_(self.head.bias)
        self.last_fc = nn.Linear(n_tokens, 1) # b f d -> b 1 d
        attenuated_kaiming_uniform_(self.last_fc.weight)
        # nn_init.zeros_(self.last_fc.bias)


    def _get_kv_compressions(self, layer):
        return (
            (self.shared_kv_compression, self.shared_kv_compression)
            if self.shared_kv_compression is not None
            else (layer['key_compression'], layer['value_compression'])
            if 'key_compression' in layer and 'value_compression' in layer
            else (layer['key_compression'], layer['key_compression'])
            if 'key_compression' in layer
            else (None, None)
        )

    def _start_residual(self, x, layer, norm_idx):
        x_residual = x
        if self.prenormalization:
            norm_key = f'norm{norm_idx}'
            if norm_key in layer:
                x_residual = layer[norm_key](x_residual)
        return x_residual

    def _end_residual(self, x, x_residual, layer, norm_idx):
        if self.residual_dropout:
            x_residual = F.dropout(x_residual, self.residual_dropout, self.training)
        x = x + x_residual
        if not self.prenormalization:
            x = layer[f'norm{norm_idx}'](x)
        return x

    def forward(self, x_num: Tensor, x_cat: ty.Optional[Tensor], mixup: bool=False, beta=0.5, mtype='feat_mix') -> Tensor:
        assert x_cat is not None
        if mtype == 'niave_mix': # naive mixup
            x_num, feat_masks, shuffled_ids = lib.mixup_data(x_num, beta=beta)
        x = self.tokenizer(x_num) # TODO: replace with PWE  b. f -> b,f ,d
        if mixup and mtype != 'niave_mix':
            mixup_func = {
                'feat_mix': lib.batch_feat_shuffle,
                'hidden_mix': lib.batch_dim_shuffle,
            }[mtype]
            x, feat_masks, shuffled_ids = mixup_func(x, beta=beta)

        for layer_idx, layer in enumerate(self.layers):
            layer = ty.cast(ty.Dict[str, nn.Module], layer)

            x_residual = self._start_residual(x, layer, 0)
            x_residual = layer['attention'](
                x_residual,
                x_residual,
                *self._get_kv_compressions(layer),
            )
            x = self._end_residual(x, x_residual, layer, 0)

            # reglu
            x_residual = self._start_residual(x, layer, 1)
            x_residual = layer['linear0'](x_residual)
            x_residual = self.activation(x_residual)
            x = self._end_residual(x, x_residual, layer, 1)

        x = self.last_fc(x.transpose(1,2))[:,:,0] # b f d -> b d
        if self.last_normalization is not None:
            x = self.last_normalization(x)
        x = self.last_activation(x) # TODO: before last_fc？
        x = self.head(x)
        x = x.squeeze(-1)

        if mixup:
            return x, feat_masks, shuffled_ids
        return x

In [ ]:
# Import required libraries

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
import math
import os
import sys
from pathlib import Path

# Clone ExcelFormer repository if not already present
if not os.path.exists('ExcelFormer'):
    !git clone https://github.com/WhatAShot/ExcelFormer.git

# Add ExcelFormer to Python path
repo_path = str(Path('ExcelFormer').absolute())
if repo_path not in sys.path:
    sys.path.append(repo_path)

# Import ExcelFormer implementation
# from ExcelFormer.bin.excel_former import ExcelFormer
from ExcelFormer.lib import get_activation_fn

# Read and process the data
df = pd.read_csv('/content/sample_data/corrected_permacts.csv', index_col=0)

# Identify categorical and numerical columns
categorical_columns = [
    'Genre', 'ContentRating', 'CurrentVersion', 'AndroidVersion',
    'DeveloperCategory', 'lowest_android_version', 'highest_android_version'
]
numerical_columns = [col for col in df.columns if col not in categorical_columns + ['status', 'pkgname']]

# Process categorical features
label_encoders = {}
cat_cardinalities = []  # Store number of unique values for each categorical feature
encoded_cat_features = []

for col in categorical_columns:
    le = LabelEncoder()
    encoded_features = le.fit_transform(df[col].astype(str))
    encoded_cat_features.append(encoded_features)
    label_encoders[col] = le
    cat_cardinalities.append(len(le.classes_))
    print(f"Feature {col} has {len(le.classes_)} unique values")

# Convert to numpy arrays
X_cat = np.stack(encoded_cat_features, axis=1)
X_num = df[numerical_columns].values
y = df['status'].values

print(f"Number of numerical columns: {len(numerical_columns)}")
print(f"Numerical features shape: {X_num.shape}")
print(f"Categorical features shape: {X_cat.shape}")

# Split the data
X_num_train, X_num_test, X_cat_train, X_cat_test, y_train, y_test = train_test_split(
    X_num, X_cat, y, test_size=0.2, random_state=42, stratify=y
)

# Scale numerical features
scaler = StandardScaler()
X_num_train = scaler.fit_transform(X_num_train)
X_num_test = scaler.transform(X_num_test)

# Convert to PyTorch tensors
X_num_train_tensor = torch.FloatTensor(X_num_train)
X_cat_train_tensor = torch.LongTensor(X_cat_train)
y_train_tensor = torch.FloatTensor(y_train)

X_num_test_tensor = torch.FloatTensor(X_num_test)
X_cat_test_tensor = torch.LongTensor(X_cat_test)
y_test_tensor = torch.FloatTensor(y_test)

# Create data loaders
train_dataset = TensorDataset(X_num_train_tensor, X_cat_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_num_test_tensor, X_cat_test_tensor, y_test_tensor)

batch_size = 256
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

# Initialize ExcelFormer model
model = ExcelFormer(
    # Tokenizer settings
    d_numerical=len(numerical_columns),  # Number of numerical features
    categories=cat_cardinalities,  # List of cardinalities for each categorical feature
    token_bias=True,

    # Transformer settings
    n_layers=3,
    d_token=256,
    n_heads=8,
    attention_dropout=0.3,
    ffn_dropout=0.3,
    residual_dropout=0.0,
    prenormalization=True,

    # Linformer settings
    kv_compression=None,
    kv_compression_sharing=None,

    # Output settings
    d_out=1,  # Binary classification
    init_scale=0.1,
)

# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
model = model.to(device)

# Set up loss function and optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

# Training loop
n_epochs = 100
best_test_auc = 0

print("Starting training...")

for epoch in range(n_epochs):
    # Training phase
    model.train()
    train_loss = 0
    for batch_num, batch_cat, batch_y in train_loader:
        batch_num = batch_num.to(device)
        batch_cat = batch_cat.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        outputs = model(batch_num, batch_cat)
        loss = criterion(outputs, batch_y)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # Evaluation phase
    model.eval()
    test_loss = 0
    predictions = []
    true_labels = []

    with torch.no_grad():
        for batch_num, batch_cat, batch_y in test_loader:
            batch_num = batch_num.to(device)
            batch_cat = batch_cat.to(device)
            batch_y = batch_y.to(device)

            outputs = model(batch_num, batch_cat)
            loss = criterion(outputs, batch_y)
            test_loss += loss.item()

            predictions.extend(torch.sigmoid(outputs).cpu().numpy())
            true_labels.extend(batch_y.cpu().numpy())

    # Calculate AUC
    from sklearn.metrics import roc_auc_score
    test_auc = roc_auc_score(true_labels, predictions)

    # Print metrics
    print(f'Epoch {epoch+1}/{n_epochs}')
    print(f'Train Loss: {train_loss/len(train_loader):.4f}')
    print(f'Test Loss: {test_loss/len(test_loader):.4f}')
    print(f'Test AUC: {test_auc:.4f}')
    print('-'*50)

    # Save best model
    if test_auc > best_test_auc:
        best_test_auc = test_auc
        torch.save(model.state_dict(), 'best_excelformer.pth')

print(f'Best Test AUC: {best_test_auc:.4f}')